# Part 2 – Curated Dataset

## Step 1: Load and Combine Raw Excel Files

The raw data is stored as separate Excel files for different years (2022–2025).

In this step, I load all Excel files from the raw data directory, add a year column based on the filename, and combine them into a single dataframe for further cleaning and harmonization.

Combining the datasets into one dataframe makes it easier to:
- inspect structural differences between years
- identify inconsistencies
- harmonize column names and data types
- perform unified cleaning and transformation

In [9]:
# This makes it so that jupyter wont just output the last line, but everything that gets outputted.
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import numpy as np

from pathlib import Path
from pydantic import BaseModel


In [10]:
# path to raw data
raw_path = Path("../data/raw")

# find all Excel files
excel_files = raw_path.glob("*.xlsx")

In [11]:
# helper functions
def load_excel(file, sheet_name):
    print(f"File: {file}")
    year = int(file.stem[-4:])
    if year == 2022:
        header_row = 0
    elif year == 2025:
        header_row = 6
    else:
        header_row = 5

    try:
        df = pd.read_excel(file, sheet_name=sheet_name, header=header_row)

        df["source_year"] = year
        df["source_file"] = Path(file).name
        df["source_sheet"] = sheet_name

        return df

    except Exception as e:
        print(f"Failed to load {file.name}: {e}")

        return None


def load_all_years(sheet_name):
    dfs = {}

    for file in excel_files:
        year = file.stem[-4:]
        dfs[year] = load_excel(file, sheet_name)

    return dfs

def check_schema(dfs):
    for year, df in dfs.items():
        print(f"\n===== DF {year} =====")
        print(f"Shape: {df.shape}")
        print(df.columns.tolist())


## Step 2: Comparing Table Structures Across Years

To prepare for harmonization, I inspected the structure of Tabell 3 across all years.

The inspection included:
- dataset dimensions
- column names
- structural differences between years

The comparison revealed that the 2022 dataset contains fewer columns than the datasets from 2023–2025.

This indicates that schema harmonization is required before the datasets can be combined into a unified curated dataset.

In [12]:
# Inspect each dataframe
dfs= load_all_years("Tabell 3")

check_schema(dfs)

File: ../data/raw/resultat-ansokningsomgang-2022.xlsx
File: ../data/raw/resultat-ansokningsomgang-2023.xlsx
File: ../data/raw/resultat-ansokningsomgang-2024.xlsx
File: ../data/raw/resultat-ansokningsomgang-2025.xlsx

===== DF 2022 =====
Shape: (1207, 19)
['Utbildningsområde', 'Utbildningsnamn', 'Beslut', 'Diarienummer', 'Län', 'Kommun', 'Flera kommuner', 'Antal kommuner', 'YH-poäng', 'Studieform', 'Studietakt %', 'Typ av examen', 'Utbildningsanordnare administrativ enhet', 'Huvudmannatyp', 'Sökta utbildningsomgångar', 'Beviljade utbildningsomgångar', 'source_year', 'source_file', 'source_sheet']

===== DF 2023 =====
Shape: (1258, 31)
['Utbildningsområde', 'SUN5 inriktning', 'SUN5 inriktning namn', 'Utbildningsnamn', 'Beslut', 'Diarienummer', 'Flera kommuner', 'Antal kommuner', 'Län', 'Kommun', 'YH-poäng', 'Studieform', 'Studietakt %', 'Typ av examen', 'SeQF nivå', 'Smalt yrkesområde', 'Utbildningsanordnare administrativ enhet', 'Huvudmannatyp', 'Sökta utbildningsomgångar', 'Beviljade u

## Step 3: Define Target Schema

Based on the schema comparison, a common target schema was defined for the curated dataset.

A column mapping was also created to standardize columns with inconsistent names across years and align them with the target schema.

In [13]:
TARGET_COLUMNS = [
    "source_year",
    "source_file",
    "source_sheet",
    "diarienummer",
    "utbildningsnamn",
    "utbildningsomrade",
    "beslut",
    "beslut_normalized",
    "kommun",
    "lan",
    "yh_poang",
    "studieform",
    "studietakt_procent",
    "utbildningsanordnare",
    "huvudmannatyp",
    "sun5_inriktning",
    "sun5_inriktning_namn",
    "seqf_niva",
    "smalt_yrkesomrade"
]

COLUMN_MAPPING = {
    "studietakt_%": "studietakt_procent",
    "utbildningsanordnare_administrativ_enhet": "utbildningsanordnare",
}

## Step 4: Column Standardization & Basic Cleaning

After defining the target schema, the next step was to standardize and clean the datasets across all years.

Because the historical Excel files were created in different years, the column names and text formatting were not fully consistent. To prepare the datasets for harmonization and integration, a basic cleaning process was applied.

The cleaning process included:

- converting column names to lowercase
- replacing spaces with underscores
- removing special characters and parentheses
- trimming whitespace from string values
- preparing column names for mapping into the target schema

This step improves consistency and reduces potential issues during schema harmonization and dataset merging.

In [14]:
def clean_column_name(col):
    return (
        str(col)
        .strip()
        .lower()
        .replace("å", "a")
        .replace("ä", "a")
        .replace("ö", "o")
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("-", "_")
    )


def clean_string_values(df):
    df = df.copy()
    
    for col in df.select_dtypes(include=["object","string"]).columns:
        df[col]=df[col].str.strip()

    return df

# clean all years
standardized_dfs ={}

for year, df in dfs.items():
    df = df.copy()

    # clean column names
    df.columns =[clean_column_name(col) for col in df.columns]

    # clean string values
    df= clean_string_values(df)

    standardized_dfs[year] =df


# check result
check_schema(standardized_dfs)


===== DF 2022 =====
Shape: (1207, 19)
['utbildningsomrade', 'utbildningsnamn', 'beslut', 'diarienummer', 'lan', 'kommun', 'flera_kommuner', 'antal_kommuner', 'yh_poang', 'studieform', 'studietakt_%', 'typ_av_examen', 'utbildningsanordnare_administrativ_enhet', 'huvudmannatyp', 'sokta_utbildningsomgangar', 'beviljade_utbildningsomgangar', 'source_year', 'source_file', 'source_sheet']

===== DF 2023 =====
Shape: (1258, 31)
['utbildningsomrade', 'sun5_inriktning', 'sun5_inriktning_namn', 'utbildningsnamn', 'beslut', 'diarienummer', 'flera_kommuner', 'antal_kommuner', 'lan', 'kommun', 'yh_poang', 'studieform', 'studietakt_%', 'typ_av_examen', 'seqf_niva', 'smalt_yrkesomrade', 'utbildningsanordnare_administrativ_enhet', 'huvudmannatyp', 'sokta_utbildningsomgangar', 'beviljade_utbildningsomgangar', 'sokta_platser_per_utbildningsomgang', 'sokta_platser_totalt', 'beviljade_platser_utbildningsomgang_1', 'beviljade_platser_utbildningsomgang_2', 'beviljade_platser_utbildningsomgang_3', 'beviljad

## Step 5: Schema Harmonization

After standardizing the column names, the datasets still contained structural differences between years. In particular, the 2022 dataset included fewer columns than the datasets from 2023–2025.

To create a unified curated dataset, the schemas were harmonized into a common target structure.

The harmonization process included:

- aligning all datasets to the predefined target schema
- adding missing columns with null values
- keeping only relevant columns
- ensuring a consistent column order across all years
- adding source metadata columns for traceability

This step ensures that all yearly datasets can be safely combined into a single curated dataset.

In [15]:
# harmonize single dataframe
def harmonize_schema(df, target_columns):

    df = df.copy()

    # rename source columns to target names
    df = df.rename(columns=COLUMN_MAPPING)

    # create normalized decision from beslut
    if "beslut" in df.columns:
        df["beslut_normalized"] = df["beslut"].astype("string").str.strip().str.lower()

    # add missing columns
    for col in target_columns:
        if col not in df.columns:
            df[col] = pd.NA

    # keep only target columns
    df = df[target_columns]

    return df


# harmonize all dataframes
harmonized_dfs = {}
for year, df in standardized_dfs.items():
    harmonized_df = harmonize_schema(df=df, target_columns=TARGET_COLUMNS)

    harmonized_dfs[year] = harmonized_df


check_schema(harmonized_dfs)



===== DF 2022 =====
Shape: (1207, 19)
['source_year', 'source_file', 'source_sheet', 'diarienummer', 'utbildningsnamn', 'utbildningsomrade', 'beslut', 'beslut_normalized', 'kommun', 'lan', 'yh_poang', 'studieform', 'studietakt_procent', 'utbildningsanordnare', 'huvudmannatyp', 'sun5_inriktning', 'sun5_inriktning_namn', 'seqf_niva', 'smalt_yrkesomrade']

===== DF 2023 =====
Shape: (1258, 19)
['source_year', 'source_file', 'source_sheet', 'diarienummer', 'utbildningsnamn', 'utbildningsomrade', 'beslut', 'beslut_normalized', 'kommun', 'lan', 'yh_poang', 'studieform', 'studietakt_procent', 'utbildningsanordnare', 'huvudmannatyp', 'sun5_inriktning', 'sun5_inriktning_namn', 'seqf_niva', 'smalt_yrkesomrade']

===== DF 2024 =====
Shape: (1272, 19)
['source_year', 'source_file', 'source_sheet', 'diarienummer', 'utbildningsnamn', 'utbildningsomrade', 'beslut', 'beslut_normalized', 'kommun', 'lan', 'yh_poang', 'studieform', 'studietakt_procent', 'utbildningsanordnare', 'huvudmannatyp', 'sun5_inr

## Step 6: Combine Harmonized Datasets

After harmonizing the schemas across all years, the yearly datasets were combined into a single curated dataset.

Because all datasets now shared the same column structure and naming convention, they could be safely merged into one unified table.

The combined dataset represents application-related information from multiple years in a consistent and analysis-ready format.

This curated dataset will later be used for validation, database integration, and API development.

In [16]:
curated_df = pd.concat(harmonized_dfs.values(), ignore_index=True)

curated_df.info()
curated_df.sample(20)


<class 'pandas.DataFrame'>
RangeIndex: 4921 entries, 0 to 4920
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   source_year           4921 non-null   int64 
 1   source_file           4921 non-null   str   
 2   source_sheet          4921 non-null   str   
 3   diarienummer          4921 non-null   str   
 4   utbildningsnamn       4921 non-null   str   
 5   utbildningsomrade     4921 non-null   str   
 6   beslut                4921 non-null   str   
 7   beslut_normalized     4921 non-null   string
 8   kommun                4921 non-null   str   
 9   lan                   4921 non-null   str   
 10  yh_poang              4921 non-null   int64 
 11  studieform            4921 non-null   str   
 12  studietakt_procent    4921 non-null   int64 
 13  utbildningsanordnare  4921 non-null   str   
 14  huvudmannatyp         4921 non-null   str   
 15  sun5_inriktning       3714 non-null   object
 16 

,source_year,source_file,source_sheet,diarienummer,utbildningsnamn,utbildningsomrade,beslut,beslut_normalized,kommun,lan,yh_poang,studieform,studietakt_procent,utbildningsanordnare,huvudmannatyp,sun5_inriktning,sun5_inriktning_namn,seqf_niva,smalt_yrkesomrade
1925,2023,resultat-ansokningsomgang-2023.xlsx,Tabell 3,MYH 2023/4404,Tjänstedesigner,Journalistik och information,Avslag,avslag,Stockholm,Stockholm,420,Bunden,100,Stockholm School of Business,Privat,321by,Övriga utbildningar medie- och kommunikationsv...,5.0,Nej
1136,2022,resultat-ansokningsomgang-2022.xlsx,Tabell 3,MYH 2022/4732,"Produktionstekniker, inriktning ständiga förbä...",Teknik och tillverkning,Beviljad,beviljad,Eslöv,Skåne,210,Distans,75,"Eslövs kommun, Yrkeshögskolan i Eslöv",Kommun,<NA>,<NA>,<NA>,<NA>
1669,2023,resultat-ansokningsomgang-2023.xlsx,Tabell 3,MYH 2023/3938,Lönespecialist med systemfokus,"Ekonomi, administration och försäljning",Avslag,avslag,Helsingborg,Skåne,400,Bunden,100,TUC Sweden AB - Yrkeshögskola,Privat,345ca,Utbildningar till löneadministratör,5.0,Nej
1222,2023,resultat-ansokningsomgang-2023.xlsx,Tabell 3,MYH 2023/3319,.NET Utvecklare,Data/IT,Beviljad,beviljad,Göteborg,Västra Götaland,410,Bunden,100,Plushögskolan AB - Teknikhögskolan Väst,Privat,481ac,Utbildningar inom systemhantering och programm...,5.0,Nej
505,2022,resultat-ansokningsomgang-2022.xlsx,Tabell 3,MYH 2022/4520,Hud- och skönhetsterapeut med inriktning mot h...,Friskvård och kroppsvård,Avslag,avslag,Göteborg,Västra Götaland,200,Bunden,100,BC Academy,Privat,<NA>,<NA>,<NA>,<NA>
2174,2023,resultat-ansokningsomgang-2023.xlsx,Tabell 3,MYH 2023/3691,Drift- och fastighetstekniker,Samhällsbyggnad och byggteknik,Beviljad,beviljad,Flera kommuner,Flera kommuner,380,Bunden,100,Folkuniversitetet - Kursverksamheten vid Stock...,Privat,582xa,Utbildningar till fastighetstekniker,5.0,Nej
1018,2022,resultat-ansokningsomgang-2022.xlsx,Tabell 3,MYH 2022/4845,Certifierad produktionstekniker,Teknik och tillverkning,Beviljad,beviljad,Piteå,Norrbotten,230,Distans,100,Lernia Utbildning AB,Privat,<NA>,<NA>,<NA>,<NA>
2065,2023,resultat-ansokningsomgang-2023.xlsx,Tabell 3,MYH 2023/3527,Miljösamordnare Hållbara Byggnader,Samhällsbyggnad och byggteknik,Beviljad,beviljad,Flera kommuner,Flera kommuner,400,Bunden,100,KYH AB,Privat,582cc,Utbildningar till byggnadsarbete mot miljö,5.0,Nej
4476,2025,resultat-ansokningsomgang-2025.xlsx,Tabell 3,MYH 2025/4106,Learning & Development Specialist,Pedagogik och undervisning,Beviljad,beviljad,Stockholm,Stockholm,320,Distans,100,Medieinstitutet i Sverige AB,Privat,149xy,Övriga utbildningar inom pedagogik och lärarut...,5.0,Nej
3552,2024,resultat-ansokningsomgang-2024.xlsx,Tabell 3,MYH 2024/3457,Produktionstekniker - Hållbar digitalisering,Teknik och tillverkning,Avslag,avslag,Örebro,Örebro,335,Bunden,100,Consensus Sverige AB,Privat,521ce,Utbildningar till produktionstekniker,5.0,Nej


## Step 7: Validation and Quality Checks

After combining the yearly datasets into a single curated dataset, several validation and quality checks were performed.

The purpose of these checks was to ensure that:

- the schema was consistent
- the merge operation was successful
- important fields were populated
- data types were reasonable
- duplicate rows could be identified
- missing values were understood

These checks help verify that the curated dataset is reliable and ready for further use in databases and APIs.

In [17]:
print("======= dataset overview =======")
curated_df.info()

print("======= missing values =======")
curated_df.isna().sum()

print("======= duplicates =======") 
curated_df.duplicated().sum()

print("====== inspect low-candinality columns values ======")
inspect_columns =[
  "source_year",
  "beslut_normalized",
  "studieform",
  "huvudmannatyp",
  "seqf_niva"
]

for col in inspect_columns:
  print(f"====== {col} ======")
  print(curated_df[col].value_counts(dropna=False))


======= dataset overview =======
<class 'pandas.DataFrame'>
RangeIndex: 4921 entries, 0 to 4920
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   source_year           4921 non-null   int64 
 1   source_file           4921 non-null   str   
 2   source_sheet          4921 non-null   str   
 3   diarienummer          4921 non-null   str   
 4   utbildningsnamn       4921 non-null   str   
 5   utbildningsomrade     4921 non-null   str   
 6   beslut                4921 non-null   str   
 7   beslut_normalized     4921 non-null   string
 8   kommun                4921 non-null   str   
 9   lan                   4921 non-null   str   
 10  yh_poang              4921 non-null   int64 
 11  studieform            4921 non-null   str   
 12  studietakt_procent    4921 non-null   int64 
 13  utbildningsanordnare  4921 non-null   str   
 14  huvudmannatyp         4921 non-null   str   
 15  sun5_inriktning 

source_year                0
source_file                0
source_sheet               0
diarienummer               0
utbildningsnamn            0
utbildningsomrade          0
beslut                     0
beslut_normalized          0
kommun                     0
lan                        0
yh_poang                   0
studieform                 0
studietakt_procent         0
utbildningsanordnare       0
huvudmannatyp              0
sun5_inriktning         1207
sun5_inriktning_namn    1207
seqf_niva               1260
smalt_yrkesomrade       1207
dtype: int64

======= duplicates =======


np.int64(0)

====== inspect low-candinality columns values ======
====== source_year ======
source_year
2024    1272
2023    1258
2022    1207
2025    1184
Name: count, dtype: int64
====== beslut_normalized ======
beslut_normalized
avslag        3217
beviljad      1703
återkallad       1
Name: count, dtype: Int64
====== studieform ======
studieform
Bunden     2572
Distans    2349
Name: count, dtype: int64
====== huvudmannatyp ======
huvudmannatyp
Privat     4073
Kommun      779
Region       60
Statlig       9
Name: count, dtype: int64
====== seqf_niva ======
seqf_niva
5.0     3591
<NA>    1207
6.0       70
NaN       53
Name: count, dtype: int64


## Step 8: Export Curated Dataset

After validation and quality checks, the curated dataset was exported for further use.

The exported file represents the cleaned, harmonized, and combined dataset. It can later be loaded into a database and used as the data source for an API.

The dataset was exported as a CSV file to make it easy to inspect and reuse in the next part of the assignment.

In [18]:
output_path = Path("../data/curated")
output_path.mkdir(parents= True, exist_ok=True)

curated_df.to_csv(
  output_path/"curated_applications.csv",
  index=False
)


## Reflection

Working with historical Excel files highlighted several common data engineering challenges, including inconsistent schemas, missing columns, and differences in formatting between years.

Defining a target schema before merging the datasets made the harmonization process more structured and easier to manage.

This assignment also demonstrated the importance of cleaning and validating data before loading it into a database or exposing it through an API.